In [187]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main
CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [189]:
from src.models.MatrixFactorization import MF, UMF
from src.graphs import random_k_out_graph, create_graph
from src.users import User
from src.training.decentralized import decentralized_train_loop, decentralized_validate_loop, decentralized_train_n_epochs
from src.data_utils import create_batched_dataloaders, create_dataloader

In [191]:
#Make data sample iterable
from torch.utils.data import Dataset, DataLoader, TensorDataset, Sampler
import torch
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import SGD

from collections import Counter
import networkx as nx
from networkx.generators.classic import empty_graph
from networkx.utils import discrete_sequence, py_random_state, weighted_choice

import seaborn as sns

In [193]:
from tqdm.auto import tqdm
from src.training.train_utils import EarlyStopper

In [164]:
def share_gradient(user, users, graph, item_id=None):

    commute = 0 
    commute_cost = 0
    user_neighbors = graph.adj[user.id]
    # user_gradients = {
    #     name: param.grad for name, param in user.model.named_parameters() if "item" in name
    # }
    item_parameter_names = ["item_factors.weight", "item_bias.weight"]
    if user.model_name == "gmf":
        item_parameter_names.append(f"item_layer_dict.item{item_id}.weight")
        item_parameter_names.append(f"item_layer_dict.item{item_id}.bias")
    elif user.model_name == "gmf_shared":
        item_parameter_names.append("item_layers.weight")
        item_parameter_names.append("item_layers.bias")

    user_gradients = {name: user.model.get_parameter(name).grad for name in item_parameter_names}
    # print(f"Number of elements being shared: {sum([g.numel() for _, g in user_gradients.items()])}")
    commute_cost = sum([g.numel() for _, g in user_gradients.items()])

    for neighbor_id in user_neighbors:
        neighbor = users[neighbor_id]
        neighbor_model = neighbor.model
        neighbor_optimizer = neighbor.optimizer
        neighbor_model.zero_grad() # required - diverges without this line
        commute = commute + 1
        for name in item_parameter_names:
            param = neighbor_model.get_parameter(name)
            param.grad = user_gradients[name].clone()
            
        # for name, param in neighbor_model.named_parameters():
        #     # print(name)
        #     if "item" in name:
        #         param.grad = user_gradients[name]  # Assign gradients to the neighbors
        neighbor_optimizer.step()  # Neighbors' update

    return commute, commute_cost


## Parameter Setting

In [166]:
model = "umf"
val_loader_type = "rs"
userprop = 0.6
n_factors = 30
sparse = False
batch_size = 10
graph_seed = 1
n_epochs = 50

para_vec= {
  "scalefree_userprop" : [0.006721468985407216, 0.3793755748581348, 0.7023494584199832],
  "scalefree_urs" : [0.00797255113179729, 0.7291631699209506, 0.7649689575684868],
  "scalefree_rs" : [0.043245636749499355, 0.24293301237845355, 0.6590721600407826],
  "scalefree_oaat" : [0.014505446034196021, 0.1281494707675557, 0.3063931184178566],
    
  "cycle_userprop" : [0.03448020025507248, 0.1530360406099725, 0.3265046312442892],
  "cycle_urs" : [0.015085184891905544, 0.32597756888723617, 0.9165691479123227],
  "cycle_rs" : [0.04518354114581989, 0.07432773840871296, 0.5104116722654509],
  "cycle_oaat" : [0.006051947990064438, 0.407449910177748, 0.6941867781038726],
    
  "random2_userprop" : [0.05973335259492166, 0.20270185084925238, 0.1],
  "random2_urs" : [0.03871364416669273, 0.14214480688557163, 0.4403378739685112],
  "random2_rs" : [0.03871364416669273, 0.14214480688557163, 0.01],
  "random2_oaat" : [0.012098247288774554, 0.051267232285266244, 0.5034632200402083],

  "random5_userprop" : [0.01214468819649195, 0.16071055871166323, 0.8930612583507401],
  "random5_urs" : [0.04664261576162963, 0.2261414992421005, 0.3645222958218734],
  "random5_rs" : [0.01864856189846265, 0.07043500222618476, 0.850748837624225],
  "random5_oaat" : [0.004358629931177893, 0.27784542450084454, 0.41295161556157467]
    
}

## Main

In [173]:
search_space = ["random2_userprop", "random2_urs", "random2_rs", "random2_oaat"]
model = "umf"
val_loader_type = "rs"
userprop = 0.6
n_factors = 30
sparse = False
batch_size = 10
graph_seed = 1
n_epochs = 150
test_vec = []
commute_vec = []
commute_cost_vec = []
gtypes, dl_types = map(list, zip(*map(lambda x:x.split('_'), search_space)))  

torch.manual_seed(0)


In [177]:
train_df = pd.read_csv("dataset/ml-1m_train.csv")
test_df = pd.read_csv("dataset/ml-1m_test.csv")
n_users = train_df['user_id'].nunique()
n_items = train_df['item_id'].nunique() 

train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=0)
train_data_loader = create_dataloader(
    df=train_df, dl_type=train_loader_type, batch_size=batch_size, p=0.6
    )
val_data_loader = create_dataloader(df=val_df, dl_type=val_loader_type)
test_data_loader = create_dataloader(df=test_df, dl_type=val_loader_type)

users = {}
for i in tqdm(range(n_users)):
    # model = MF(n_users=n_users, n_items=n_items)
    user_model = UMF(n_items, n_factors = n_factors, sparse = sparse)
    # model = GeneralizedMFOneLayer(n_users=n_users, n_items=n_items)
    users[i] = User(id=i, model=user_model, optimizer=SGD(user_model.parameters(), lr=lr, weight_decay=weight_decay,momentum=mom), model_name = model)

graph = random_k_out_graph(n=n_users, k=2, seed=graph_seed)  


  0%|          | 0/6040 [00:00<?, ?it/s]

In [183]:
print(n_users, n_items)

6040 3664


In [175]:
time_table = {}
rmse_table = {}
communte_table = {}
test_table = {}
for i in np.arange(len(dl_types)):

    break_status = True

    if i == 0 or i == 1:
        break_status = False
    
    train_loader_type = dl_types[i]
    graph_type = "random_2_out"#gtypes[i]
    print(train_loader_type)
    temp_para = para_vec[search_space[i]]
    lr = temp_para[0]
    weight_decay = temp_para[1]
    mom = temp_para[2]
    print(f"lr : {lr} | wd : {weight_decay} | mom : {mom}")
     
    train_df = pd.read_csv("dataset/ml-1m_train.csv")
    test_df = pd.read_csv("dataset/ml-1m_test.csv")
    n_users = train_df['user_id'].nunique()
    n_items = train_df['item_id'].nunique() 

    train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=0)
    train_data_loader = create_dataloader(
        df=train_df, dl_type=train_loader_type, batch_size=batch_size, p=0.6
        )
    val_data_loader = create_dataloader(df=val_df, dl_type=val_loader_type)
    test_data_loader = create_dataloader(df=test_df, dl_type=val_loader_type)

    users = {}
    for i in tqdm(range(n_users)):
        # model = MF(n_users=n_users, n_items=n_items)
        user_model = UMF(n_items, n_factors = n_factors, sparse = sparse)
        # model = GeneralizedMFOneLayer(n_users=n_users, n_items=n_items)
        users[i] = User(id=i, model=user_model, optimizer=SGD(user_model.parameters(), lr=lr, weight_decay=weight_decay,momentum=mom), model_name = model)
    
    graph = random_k_out_graph(n=n_users, k=2, seed=graph_seed)  
    
    train_losses, val_losses, time_per_epoch, commutes = decentralized_train_n_epochs(
        user_models=users,
        train_loader=train_data_loader,
        val_loader=val_data_loader,
        epochs=n_epochs,
        graph=graph,
        break_gate = break_status
        )  
    test_loss = decentralized_validate_loop(users, test_data_loader)
    
    time_table[train_loader_type] = time_per_epoch
    rmse_table[train_loader_type] = val_losses
    communte_table[train_loader_type] = commutes["commute"]
    test_table[train_loader_type] = test_loss

oaat
lr : 0.012098247288774554 | wd : 0.051267232285266244 | mom : 0.5034632200402083


  0%|          | 0/6040 [00:00<?, ?it/s]

KeyboardInterrupt: 

Exception ignored in: 'zmq.backend.cython._zmq.Frame.__del__'
Traceback (most recent call last):
  File "_zmq.py", line 160, in zmq.backend.cython._zmq._check_rc
KeyboardInterrupt: 


  0%|          | 0/600124 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [49]:
df = pd.DataFrame.from_dict(time_table, orient="index")
df.index.name = "epoch"
df.reset_index(inplace=True)
df.to_csv("time_table.csv", index=False)

In [51]:
df = pd.DataFrame.from_dict(rmse_table, orient="index")
df.index.name = "epoch"
df.reset_index(inplace=True)
df.to_csv("rmse_table.csv", index=False)

In [53]:
df = pd.DataFrame.from_dict(communte_table, orient="index")
df.index.name = "epoch"
df.reset_index(inplace=True)
df.to_csv("commute_table.csv", index=False)

In [55]:
df = pd.DataFrame.from_dict( test_table, orient="index")
df.index.name = "epoch"
df.reset_index(inplace=True)
df.to_csv("test_loss.csv", index=False)

In [43]:
test_table

{'userprop': 1.0326436, 'urs': 1.0726097, 'rs': 0.8873686}

In [47]:
rmse_table

{'userprop': [5.4070387,
  4.680201,
  4.068043,
  3.5948944,
  3.2363346,
  2.9485931,
  2.6992452,
  2.4932868,
  2.32649,
  2.176166,
  2.0513303,
  1.9439313,
  1.850324,
  1.7738324,
  1.7069722,
  1.638486,
  1.5828327,
  1.5337276,
  1.4933298,
  1.4444525,
  1.415324,
  1.3822877,
  1.3514019,
  1.328545,
  1.3024039,
  1.2809696,
  1.2574874,
  1.2420506,
  1.2279034,
  1.2083547,
  1.1956851,
  1.1833876,
  1.1656476,
  1.1541014,
  1.1440527,
  1.1354212,
  1.1230615,
  1.1166295,
  1.1114291,
  1.1017962,
  1.09476,
  1.0850922,
  1.080807,
  1.0730675,
  1.0655136,
  1.0609467,
  1.0577775,
  1.0516386,
  1.0478287,
  1.0436031,
  1.036446],
 'urs': [5.472488,
  5.024299,
  4.507145,
  4.0540485,
  3.65595,
  3.3168314,
  3.0358021,
  2.794904,
  2.5876482,
  2.419825,
  2.2600646,
  2.1403859,
  2.0293293,
  1.9336144,
  1.8461276,
  1.7807451,
  1.7132215,
  1.6520919,
  1.6037585,
  1.5558846,
  1.5165575,
  1.4802095,
  1.4494642,
  1.4168093,
  1.3870286,
  1.364011,
